# PrivateLocalAgent · 评委启动（Notebook）

**正确打开方式：Radeon Cloud → Open Notebook → 打开本文件 → 按顺序运行单元格。**

界面嵌在 JupyterLab 内，**不需要** `127.0.0.1:7900` / Cloudflare / rc-tunnel。

推荐问题与 Demo 视频、`scripts/demo_judge.py`、`START_HERE.md` **完全一致**。

视频：https://github.com/Wyf66669/Radeon-hackathon-2026-07/releases/download/demo-v1/PrivateLocalAgent_demo.mp4

运行顺序：
1. **Kernel → Restart Kernel**
2. 运行单元格 1，等到打印 `ready`
3. 运行单元格 2，出现可视化面板后点推荐问题发送

In [ ]:
# 1) 启动环境（持久化 + 模型 + 知识库 + 多代理）
import os, sys, subprocess
from pathlib import Path

def log(msg):
    print(msg, flush=True)

ROOT = Path("/workspace/Radeon-hackathon-2026-07")
if not ROOT.exists():
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

persist = Path("/workspace/persistence")
if persist.is_dir():
    os.environ.setdefault("PLA_DATA_ROOT", str(persist / "PrivateLocalAgent"))
    os.environ.setdefault("HF_HOME", str(persist / "huggingface"))
    Path(os.environ["PLA_DATA_ROOT"]).mkdir(parents=True, exist_ok=True)
    Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("USE_ROCM_AITER_ROPE_BACKEND", "0")

log("[0] pip...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "chromadb", "sentence-transformers", "pypdf", "pyyaml",
    "python-dotenv", "pydantic", "openai", "ipywidgets",
    "transformers", "accelerate", "safetensors", "sentencepiece",
    "Pillow",
])

from src.agent.agent import PrivateAgent
from src.agent.multi_agent import MultiAgentOrchestrator
from src.agent.tools import ToolRegistry
from src.apps.judge_script import ensure_judge_ocr_image
from src.config import load_settings
from src.llm.backend import build_llm
from src.memory.memory import SessionMemory
from src.privacy.audit import AuditTrail
from src.rag.store import VectorStore
from src.skills import SkillRegistry

settings = load_settings()
upload_dir = settings.resolve(settings.paths.upload_dir)
ensure_judge_ocr_image(upload_dir)

log("[1] KB...")
store = VectorStore(settings)
if store.count() == 0:
    n = store.add_directory(settings.resolve(settings.paths.sample_docs))
    log(f"    ingested: {n}")
else:
    log(f"    KB chunks: {store.count()}")

memory = SessionMemory(settings.resolve(settings.agent.memory_path))
skills = SkillRegistry(settings.resolve(settings.paths.generated_projects))
tools = ToolRegistry(store, memory, upload_dir, skill_registry=skills)
audit = AuditTrail(settings.resolve("data/memory/audit.jsonl"))

log("[2] load LLM on Radeon/ROCm...")
llm = build_llm(settings.llm)
agent = PrivateAgent(llm, tools, memory, settings.agent.max_steps, audit=audit)
orch = MultiAgentOrchestrator(agent, tools)
log("ready")
log("下一步：运行下一单元格打开可视化面板（与 Demo 视频同题）")

In [ ]:
# 2) Notebook 内可视化（无隧道）· 模式 + 推荐问题 = 视频
from src.app.notebook_visual import launch_notebook_visual

assert "orch" in globals(), "请先运行上一单元格直到出现 ready"
ui = launch_notebook_visual(orch, default_mode="chat")
# 备用：ui.ask("请假需要提前几天申请？", mode="rag")

### 可选：命令行一次跑完全部视频题目

在 Terminal：`python scripts/demo_judge.py`